# Fisher Forecast: Parameter Constraints from Angular Power Spectra

This notebook demonstrates a Fisher matrix forecast for cosmological
parameters using the data vector + covariance infrastructure.

### Steps
1. Set up tracers and data vector configuration
2. Compute the fiducial data vector and covariance
3. Compute the Fisher matrix for cosmological parameters
4. Show marginalized constraints and confidence ellipses
5. Compare constraints: WL-only vs NC-only vs 3×2pt combined

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

## 1. Define tracers and survey

In [ ]:
z = np.linspace(0.0, 3.0, 200)
nz_source = np.exp(-0.5 * ((z - 0.8) / 0.2)**2)
nz_lens = np.exp(-0.5 * ((z - 0.4) / 0.15)**2)

TRACER_CONFIGS = {
    "wl": dict(
        tracer_type="weak_lensing",
        dndz_z=z.tolist(), dndz_nz=nz_source.tolist(),
        has_shear=True,
        n_gal=10.0, sigma_epsilon=0.26,
    ),
    "nc": dict(
        tracer_type="number_counts",
        dndz_z=z.tolist(), dndz_nz=nz_lens.tolist(),
        bias_model="constant", bias_params={"a0": 1.5},
        has_rsd=False,
        n_gal=30.0,
    ),
}

FIDUCIAL = dict(
    Omega_c=0.2589, Omega_b=0.0486, h=0.6774,
    n_s=0.9667, sigma8=0.8159, Omega_k=0.0, w0=-1.0, wa=0.0,
)

F_SKY = 0.3
ELL = np.arange(10, 1001).astype(float)

from tissage_cosmique.computations.ell_binning import make_log_bins
BINS = make_log_bins(10, 1000, 15).tolist()

print(f"Survey: f_sky={F_SKY}, {len(ELL)} multipoles, 15 bins")
print(f"Tracers: WL (n_gal=10, σ_ε=0.26) + NC (n_gal=30, b=1.5)")

## 2. Compute fiducial data vector and covariance

In [ ]:
from tissage_cosmique.computations.covariance import compute_covariance_from_data_vector
from tissage_cosmique.computations.data_vector import compute_data_vector, make_data_vector_result

# 3x2pt: WL×WL + WL×NC + NC×NC
dv_config_3x2 = {
    "tracer_names": ["wl", "nc"],
    "bin_edges": BINS,
}

dv_3x2, dv_result_3x2, cov_3x2 = compute_covariance_from_data_vector(
    FIDUCIAL, TRACER_CONFIGS, dv_config_3x2, ELL, F_SKY,
)
cov_inv_3x2 = np.linalg.inv(cov_3x2)

print(f"3×2pt data vector: {len(dv_3x2)} bins")
print(f"Total S/N = {np.sqrt(dv_3x2 @ cov_inv_3x2 @ dv_3x2):.1f}")

## 3. Fisher matrix for (Omega_c, sigma8)

In [ ]:
from tissage_cosmique.computations.fisher import compute_fisher_matrix

PARAM_NAMES = ["Omega_c", "sigma8"]

def model_3x2(params):
    full = {**FIDUCIAL, **params}
    return compute_data_vector(full, TRACER_CONFIGS, dv_config_3x2, ELL)

fisher_3x2 = compute_fisher_matrix(
    model_3x2, FIDUCIAL, cov_inv_3x2,
    param_names=PARAM_NAMES,
    step_sizes={"Omega_c": 0.005, "sigma8": 0.005},
)

print("3×2pt Fisher forecast:")
for p in PARAM_NAMES:
    print(f"  σ({p}) = {fisher_3x2.sigma(p):.4f}")
    print(f"  σ_cond({p}) = {fisher_3x2.sigma_conditional(p):.4f}")

print(f"\nCorrelation matrix:")
corr = fisher_3x2.correlation_matrix()
for i, pi in enumerate(PARAM_NAMES):
    row = ", ".join(f"{corr[i, j]:+.3f}" for j in range(len(PARAM_NAMES)))
    print(f"  {pi}: [{row}]")

## 4. Confidence ellipses

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

for n_sig, alpha in [(1, 0.4), (2, 0.2)]:
    x, y = fisher_3x2.ellipse("Omega_c", "sigma8", n_sigma=n_sig)
    ax.fill(x, y, alpha=alpha, color="tab:blue")
    ax.plot(x, y, color="tab:blue", lw=1)

ax.plot(FIDUCIAL["Omega_c"], FIDUCIAL["sigma8"], "k+", ms=15, mew=2)
ax.set_xlabel(r"$\Omega_c$")
ax.set_ylabel(r"$\sigma_8$")
ax.set_title("3×2pt Fisher forecast (1σ and 2σ)")
plt.tight_layout()
plt.show()

## 5. Compare: WL-only vs NC-only vs 3×2pt

Run separate Fisher forecasts for each probe alone and the combination.

In [ ]:
# WL-only
dv_config_wl = {
    "tracer_names": ["wl"],
    "include_pairs": [["wl", "wl"]],
    "bin_edges": BINS,
}
_, _, cov_wl = compute_covariance_from_data_vector(
    FIDUCIAL, TRACER_CONFIGS, dv_config_wl, ELL, F_SKY,
)
fisher_wl = compute_fisher_matrix(
    lambda p: compute_data_vector({**FIDUCIAL, **p}, TRACER_CONFIGS, dv_config_wl, ELL),
    FIDUCIAL, np.linalg.inv(cov_wl),
    param_names=PARAM_NAMES,
    step_sizes={"Omega_c": 0.005, "sigma8": 0.005},
)

# NC-only
dv_config_nc = {
    "tracer_names": ["nc"],
    "include_pairs": [["nc", "nc"]],
    "bin_edges": BINS,
}
_, _, cov_nc = compute_covariance_from_data_vector(
    FIDUCIAL, TRACER_CONFIGS, dv_config_nc, ELL, F_SKY,
)
fisher_nc = compute_fisher_matrix(
    lambda p: compute_data_vector({**FIDUCIAL, **p}, TRACER_CONFIGS, dv_config_nc, ELL),
    FIDUCIAL, np.linalg.inv(cov_nc),
    param_names=PARAM_NAMES,
    step_sizes={"Omega_c": 0.005, "sigma8": 0.005},
)

print(f"{'Probe':<12} {'σ(Omega_c)':>12} {'σ(sigma8)':>12}")
print("-" * 38)
for name, f in [("WL only", fisher_wl), ("NC only", fisher_nc), ("3×2pt", fisher_3x2)]:
    print(f"{name:<12} {f.sigma('Omega_c'):>12.4f} {f.sigma('sigma8'):>12.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

probes = [
    ("WL only", fisher_wl, "tab:blue"),
    ("NC only", fisher_nc, "tab:green"),
    ("3×2pt", fisher_3x2, "tab:red"),
]

for label, f, color in probes:
    for n_sig in [1, 2]:
        x, y = f.ellipse("Omega_c", "sigma8", n_sigma=n_sig)
        ax.plot(x, y, color=color, lw=1.5 if n_sig == 1 else 0.8,
                label=f"{label} ({n_sig}σ)" if n_sig == 1 else None)

ax.plot(FIDUCIAL["Omega_c"], FIDUCIAL["sigma8"], "k+", ms=15, mew=2, label="Fiducial")
ax.set_xlabel(r"$\Omega_c$")
ax.set_ylabel(r"$\sigma_8$")
ax.set_title("Fisher forecast: probe comparison")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 6. Derivatives: how does the data vector change with parameters?

In [ ]:
from tissage_cosmique.computations.ell_binning import pair_key

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

dv_res = make_data_vector_result(dv_3x2, dv_config_3x2, ELL)
pair_colors = {"wl-wl": "tab:blue", "wl-nc": "tab:orange", "nc-nc": "tab:green"}
pair_labels = {"wl-wl": "WL×WL", "wl-nc": "WL×NC", "nc-nc": "NC×NC"}

for ax, param in zip(axes, PARAM_NAMES):
    deriv = fisher_3x2.derivatives[param]
    for i, (t1, t2) in enumerate(dv_res.pairs):
        key = pair_key(t1, t2)
        off = dv_res.offsets[i]
        n = dv_res.lengths[i]
        ell_eff = dv_res.get_ell(t1, t2)
        ax.plot(ell_eff, deriv[off:off + n], "o-", ms=4,
                color=pair_colors[key], label=pair_labels[key])
    ax.set_xlabel(r"$\ell_{\rm eff}$")
    ax.set_ylabel(rf"$\partial C_\ell / \partial {param}$")
    ax.set_title(rf"Data vector derivative w.r.t. {param}")
    ax.set_xscale("log")
    ax.legend(fontsize=8)
    ax.axhline(0, color="gray", lw=0.5)

plt.tight_layout()
plt.show()

## 7. Likelihood surface scan

In [ ]:
from tissage_cosmique.computations.likelihood import chi_squared

oc_range = np.linspace(0.20, 0.32, 30)
s8_range = np.linspace(0.75, 0.88, 30)
chi2_grid = np.zeros((len(s8_range), len(oc_range)))

for i, s8 in enumerate(s8_range):
    for j, oc in enumerate(oc_range):
        model = model_3x2({"Omega_c": oc, "sigma8": s8})
        chi2_grid[i, j] = chi_squared(dv_3x2, model, cov_inv_3x2)

# Subtract minimum for Δχ²
dchi2 = chi2_grid - chi2_grid.min()

fig, ax = plt.subplots(figsize=(8, 6))
cs = ax.contourf(oc_range, s8_range, dchi2, levels=[0, 2.30, 6.18, 11.83],
                 colors=["#2166ac", "#67a9cf", "#d1e5f0"], alpha=0.7)
ax.contour(oc_range, s8_range, dchi2, levels=[2.30, 6.18, 11.83],
           colors=["darkblue", "blue", "lightblue"], linewidths=1)

# Overlay Fisher ellipses
for n_sig in [1, 2, 3]:
    x, y = fisher_3x2.ellipse("Omega_c", "sigma8", n_sigma=n_sig)
    ax.plot(x, y, "r--", lw=1)

ax.plot(FIDUCIAL["Omega_c"], FIDUCIAL["sigma8"], "r+", ms=15, mew=2)
ax.set_xlabel(r"$\Omega_c$")
ax.set_ylabel(r"$\sigma_8$")
ax.set_title(r"$\Delta\chi^2$ surface (filled) vs Fisher ellipses (dashed red)")
plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated:

- **Fisher matrix** computation via numerical derivatives of the binned data vector
- **Marginalized constraints** σ(Ω_c) and σ(σ_8) from F⁻¹
- **Probe comparison**: WL-only, NC-only, and combined 3×2pt — the combination breaks degeneracies
- **Confidence ellipses** showing the Ω_c–σ_8 degeneracy direction
- **Derivative visualization** showing how each spectrum responds to parameter changes
- **Likelihood surface** Δχ² scan confirming the Fisher ellipses match the true likelihood

The Fisher forecast can use emulated data vectors instead of pyccl for
orders-of-magnitude speedup when scanning over many parameters.